# Bee Pollen CNN — Kaggle training

Before running: add the Bee Pollen dataset and the corrections CSV dataset as Inputs; select **GPU T4 x2**; enable Internet. Edit the two paths in the next cell from Kaggle's Input panel.

In [ ]:
# CHANGE DATA_DIR to the exact path displayed by Kaggle's Input panel.
DATA_DIR = '/kaggle/input/honey-bee-pollen/REPLACE_WITH_CLASS_FOLDER_PARENT'
# Phase 1: use RUN_MODE = 'original' and leave CORRECTIONS_CSV = None.
# Phase 2: upload reviewed corrections.csv as a Kaggle Input, then set RUN_MODE = 'corrected'.
RUN_MODE = 'original'  # 'original', 'corrected', or 'all'
CORRECTIONS_CSV = None

REPOSITORY_URL = 'https://github.com/Dacekey/DPL-Lab2_CNN-BeePollen.git'
REPOSITORY_DIR = '/kaggle/working/DPL-Lab2_CNN-BeePollen'
EPOCHS = 25
IMAGE_SIZE = 128
BATCH_PER_REPLICA = 32
SEED = 42


In [ ]:
# Run this first after selecting Settings -> Accelerator -> GPU T4 x2.
!nvidia-smi

import tensorflow as tf
print('TensorFlow GPUs:', tf.config.list_physical_devices('GPU'))


In [ ]:
import os
import subprocess
from pathlib import Path

assert 'REPLACE_WITH' not in DATA_DIR, 'Set DATA_DIR first.'
assert Path(DATA_DIR).is_dir(), f'DATA_DIR not found: {DATA_DIR}'
assert RUN_MODE in {'original', 'corrected', 'all'}
if RUN_MODE in {'corrected', 'all'}:
    assert CORRECTIONS_CSV and Path(CORRECTIONS_CSV).is_file(), 'Set CORRECTIONS_CSV for the corrected run.'

# Kaggle sessions start clean. On a rerun, keep the already cloned source.
if not Path(REPOSITORY_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, REPOSITORY_DIR], check=True)
else:
    print('Repository already cloned:', REPOSITORY_DIR)


In [ ]:
import shlex

command = [
    'python', f'{REPOSITORY_DIR}/src/train_lab.py',
    '--data-dir', DATA_DIR,
    '--corrections-csv', CORRECTIONS_CSV,
    '--output-dir', f'{REPOSITORY_DIR}/outputs',
    '--epochs', str(EPOCHS),
    '--image-size', str(IMAGE_SIZE),
    '--batch-per-replica', str(BATCH_PER_REPLICA),
    '--seed', str(SEED),
    '--run-mode', RUN_MODE,
]
if CORRECTIONS_CSV:
    command.extend(['--corrections-csv', CORRECTIONS_CSV])
print('Running:', ' '.join(shlex.quote(x) for x in command))
subprocess.run(command, check=True)


In [ ]:
from pathlib import Path
packages = Path(REPOSITORY_DIR) / 'outputs' / 'packages'
print('Download these after Save Version:')
for package in sorted(packages.glob('*.zip')):
    print(package, f'({package.stat().st_size / 1024 / 1024:.1f} MB)')

import pandas as pd
display(pd.read_csv(Path(REPOSITORY_DIR) / 'outputs' / 'metrics_summary.csv'))
